In [1]:
# Step 1: Check uploaded CSV files

files = notebookutils.fs.ls("Files/raw")

for file in files:
    print(file.name)


StatementMeta(, da0258a6-265e-4e29-a65a-eb131e9e5fd5, 3, Finished, Available, Finished, False)

legacy_appointments.csv
legacy_centres.csv
legacy_invoices.csv
legacy_monthly_financial_report.csv
legacy_monthly_targets.csv
legacy_payments.csv
legacy_practitioners.csv
legacy_service_catalog.csv


In [2]:

from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType
)

centres_schema = StructType([
    StructField("centre_id", IntegerType(), False),
    StructField("centre_name", StringType(), True),
    StructField("state", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("active_chairs", IntegerType(), True)
])

df_centres = (
    spark.read
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(centres_schema)
    .csv("Files/raw/legacy_centres.csv")
)

df_centres.printSchema()
display(df_centres)


StatementMeta(, da0258a6-265e-4e29-a65a-eb131e9e5fd5, 4, Finished, Available, Finished, False)

root
 |-- centre_id: integer (nullable = true)
 |-- centre_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- active_chairs: integer (nullable = true)



SynapseWidget(Synapse.DataFrame, 5beed1ec-1dff-4cab-b0a1-8dcd2e8e4f0d)

In [1]:

# Step 3: Validate and save Delta table

expected_rows = 12
actual_rows = df_centres.count()

assert actual_rows == expected_rows, (
    f"Expected {expected_rows} rows, "
    f"but found {actual_rows}"
)

assert (
    df_centres.select("centre_id").distinct().count()
    == actual_rows
), "Duplicate centre_id found"

assert (
    df_centres.filter("centre_id IS NULL").count() == 0
), "Null centre_id found"

(
    df_centres.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("legacy_centres")
)

print("SUCCESS: Delta table created")
print("Rows:", spark.table("legacy_centres").count())


StatementMeta(, dd3e8ddc-f5a2-441c-a19f-e48a2000f035, 3, Finished, Available, Finished, False)

NameError: name 'df_centres' is not defined

In [2]:

from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType
)

centres_schema = StructType([
    StructField("centre_id", IntegerType(), False),
    StructField("centre_name", StringType(), True),
    StructField("state", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("active_chairs", IntegerType(), True)
])

df_centres = (
    spark.read
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(centres_schema)
    .csv("Files/raw/legacy_centres.csv")
)

print("Rows:", df_centres.count())
display(df_centres)


StatementMeta(, dd3e8ddc-f5a2-441c-a19f-e48a2000f035, 4, Finished, Available, Finished, False)

Rows: 12


SynapseWidget(Synapse.DataFrame, ff310b9a-1c81-4a5f-b10f-60259dd470fd)

In [3]:

# Validate data
assert df_centres.count() == 12

assert (
    df_centres.select("centre_id").distinct().count()
    == 12
), "Duplicate centre_id found"

# Create Delta table
(
    df_centres.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("legacy_centres")
)

print("SUCCESS: Delta table created")
print("Rows:", spark.table("legacy_centres").count())


StatementMeta(, dd3e8ddc-f5a2-441c-a19f-e48a2000f035, 5, Finished, Available, Finished, False)

SUCCESS: Delta table created
Rows: 12


In [4]:

from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    DateType, DecimalType
)

# Define schemas based on the legacy SQL database

schemas = {

    "legacy_practitioners": [
        ("practitioner_id", IntegerType()),
        ("centre_id", IntegerType()),
        ("practitioner_code", StringType()),
        ("specialty", StringType())
    ],

    "legacy_service_catalog": [
        ("service_code", StringType()),
        ("service_name", StringType()),
        ("standard_fee_aud", DecimalType(12, 2))
    ],

    "legacy_appointments": [
        ("appointment_id", IntegerType()),
        ("centre_id", IntegerType()),
        ("practitioner_id", IntegerType()),
        ("appointment_date", DateType()),
        ("service_code", StringType()),
        ("status", StringType())
    ],

    "legacy_invoices": [
        ("invoice_id", IntegerType()),
        ("appointment_id", IntegerType()),
        ("invoice_date", DateType()),
        ("gross_patient_fee_aud", DecimalType(12, 2)),
        ("facility_service_fee_aud", DecimalType(12, 2)),
        ("invoice_status", StringType())
    ],

    "legacy_payments": [
        ("payment_id", IntegerType()),
        ("invoice_id", IntegerType()),
        ("payment_date", DateType()),
        ("amount_paid_aud", DecimalType(12, 2)),
        ("payment_method", StringType())
    ],

    "legacy_monthly_targets": [
        ("centre_id", IntegerType()),
        ("reporting_month", StringType()),
        ("appointment_target", IntegerType()),
        ("facility_fee_target_aud", DecimalType(12, 2))
    ],

    "legacy_monthly_financial_report": [
        ("centre_id", IntegerType()),
        ("reporting_month", StringType()),
        ("completed_appointments", IntegerType()),
        ("facility_service_fee_aud", DecimalType(12, 2))
    ]
}

expected_counts = {
    "legacy_practitioners": 84,
    "legacy_service_catalog": 8,
    "legacy_appointments": 7200,
    "legacy_invoices": 6198,
    "legacy_payments": 5824,
    "legacy_monthly_targets": 144,
    "legacy_monthly_financial_report": 144
}

# Read, validate and create Delta tables

for table_name, columns in schemas.items():

    print(f"Processing: {table_name}")

    schema = StructType([
        StructField(name, dtype, True)
        for name, dtype in columns
    ])

    df = (
        spark.read
        .option("header", "true")
        .option("mode", "FAILFAST")
        .option("dateFormat", "yyyy-MM-dd")
        .schema(schema)
        .csv(f"Files/raw/{table_name}.csv")
    )

    actual_count = df.count()

    assert actual_count == expected_counts[table_name], (
        f"{table_name}: Expected "
        f"{expected_counts[table_name]}, "
        f"found {actual_count}"
    )

    # Prevent writing rows with null values in required keys
    key_columns = {
        "legacy_practitioners": ["practitioner_id"],
        "legacy_service_catalog": ["service_code"],
        "legacy_appointments": ["appointment_id"],
        "legacy_invoices": ["invoice_id"],
        "legacy_payments": ["payment_id"],
        "legacy_monthly_targets":
            ["centre_id", "reporting_month"],
        "legacy_monthly_financial_report":
            ["centre_id", "reporting_month"]
    }

    for key in key_columns[table_name]:
        assert df.filter(df[key].isNull()).count() == 0, (
            f"{table_name}: Null key in {key}"
        )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print(f"SUCCESS: {table_name} - {actual_count} rows")

print("All seven Delta tables created successfully!")


StatementMeta(, dd3e8ddc-f5a2-441c-a19f-e48a2000f035, 6, Finished, Available, Finished, False)

Processing: legacy_practitioners
SUCCESS: legacy_practitioners - 84 rows
Processing: legacy_service_catalog
SUCCESS: legacy_service_catalog - 8 rows
Processing: legacy_appointments
SUCCESS: legacy_appointments - 7200 rows
Processing: legacy_invoices
SUCCESS: legacy_invoices - 6198 rows
Processing: legacy_payments
SUCCESS: legacy_payments - 5824 rows
Processing: legacy_monthly_targets
SUCCESS: legacy_monthly_targets - 144 rows
Processing: legacy_monthly_financial_report
SUCCESS: legacy_monthly_financial_report - 144 rows
All seven Delta tables created successfully!


In [5]:

tables = [
    "legacy_centres",
    "legacy_practitioners",
    "legacy_service_catalog",
    "legacy_appointments",
    "legacy_invoices",
    "legacy_payments",
    "legacy_monthly_targets",
    "legacy_monthly_financial_report"
]

for table in tables:
    count = spark.table(table).count()
    print(f"{table}: {count} rows")


StatementMeta(, dd3e8ddc-f5a2-441c-a19f-e48a2000f035, 7, Finished, Available, Finished, False)

legacy_centres: 12 rows
legacy_practitioners: 84 rows
legacy_service_catalog: 8 rows
legacy_appointments: 7200 rows
legacy_invoices: 6198 rows
legacy_payments: 5824 rows
legacy_monthly_targets: 144 rows
legacy_monthly_financial_report: 144 rows
